# S1 exploratory extension: epoch 50 → 100

This notebook warm-starts from `/kaggle/input/notebooks/dyhngg/test-rq1`. It resets optimizer/scheduler state and therefore does **not** claim equivalence to a from-scratch `T_max=100` run. Shared models resume from epoch 50; each specialized model resumes from its validation-selected source `best_epoch` and trains to global epoch 100.

In [ ]:
import os, subprocess, sys, time
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'
assert torch.cuda.device_count() >= 2, 'Select GPU T4 x2'
for index in range(torch.cuda.device_count()):
    print(f'GPU {index}: {torch.cuda.get_device_name(index)}')

## Secure clone
Create a Kaggle secret named `github_token`.

In [ ]:
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Kaggle secret github_token is missing'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy()
env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN_RUNTIME': github_token})
try:
    command = ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'] if (PROJECT_ROOT / '.git').is_dir() else ['git', 'clone', 'https://github.com/duyh80456-code/new-pruning.git', str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True)
    env.pop('GITHUB_TOKEN_RUNTIME', None)
    github_token = None
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'thop>=0.1.1', 'tabulate>=0.9'], check=True)

## Materialize and validate the epoch-50 source

In [ ]:
import importlib
import scripts.run_s1_extension as extension_runner
extension_runner = importlib.reload(extension_runner)
INPUT_ROOT = Path('/kaggle/input/notebooks/dyhngg/test-rq1')
assert INPUT_ROOT.exists(), f'Attach the notebook output: {INPUT_ROOT}'
SOURCE_ROOT = extension_runner.materialize_source(
    INPUT_ROOT, '/kaggle/working/materialized-s1-epoch-50'
)
print('Validated source:', SOURCE_ROOT)

## Preflight and freeze the extension output

In [ ]:
from datetime import datetime, timezone
import yaml
from torchvision import datasets
SOURCE_CONFIG = PROJECT_ROOT / 'configs' / 'kaggle_s1_extension_100.yaml'
config = yaml.safe_load(SOURCE_CONFIG.read_text())
assert config['experiment']['source_horizon'] == 50
assert config['experiment']['target_horizon'] == 100
assert config['training']['epochs'] == 100
assert config['experiment']['extension_learning_rate'] == 0.01
datasets.CIFAR100(root=config['dataset']['root'], train=True, download=True)
datasets.CIFAR100(root=config['dataset']['root'], train=False, download=True)
RUN_NAME = datetime.now(timezone.utc).strftime('kaggle-s1-extension-100-%Y%m%d-%H%M%S')
RUN_DIR = Path('/kaggle/working/new-pruning-outputs') / RUN_NAME
config['experiment']['output_dir'] = str(RUN_DIR)
LAUNCH_CONFIG = Path('/kaggle/working/kaggle_s1_extension_100_launch.yaml')
LAUNCH_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print('Run directory:', RUN_DIR)
print('Protocol: weights-only extension, reset SGD, LR=0.01, cosine to global epoch 100')

## Execute on T4 ×2

Epoch-level extension checkpoints are resumable inside the same `RUN_DIR`. Test evaluation starts only after all extended checkpoints are complete.

In [ ]:
importlib.reload(__import__('s1_width'))
extension_runner = importlib.reload(extension_runner)
started = time.perf_counter()
result = extension_runner.run_extension(LAUNCH_CONFIG, SOURCE_ROOT, gpu_ids=[0, 1])
print(f'Extension completed in {(time.perf_counter()-started)/3600:.2f} hours')
print(result)

## Inspect epoch 50 versus extended epoch 100

In [ ]:
import pandas as pd
from IPython.display import Markdown, display
display(Markdown((RUN_DIR / 's1_report.md').read_text()))
display(pd.read_csv(RUN_DIR / 'specialization_50_to_100_comparison.csv'))
display(pd.read_csv(RUN_DIR / 'shared_50_to_100_comparison.csv'))
display(pd.read_csv(RUN_DIR / 'representation_pattern_robustness.csv'))

In [ ]:
import shutil
required = [RUN_DIR / name for name in [
    's1_report.md', 'specialization_table.csv',
    'specialization_50_to_100_comparison.csv',
    'shared_50_to_100_comparison.csv',
    'rq1_representation_gap_table.csv',
    'representation_pattern_robustness.csv',
    'protocol/extension_provenance.json',
]]
required += [RUN_DIR / 'shared' / f'seed_{seed}' / 'checkpoint.pt' for seed in [0, 1, 2]]
required += [RUN_DIR / 'specialized' / f'seed_{seed}' / f'width_{tag}' / 'best_checkpoint.pt' for seed in [0, 1, 2] for tag in ['030', '040', '060', '080']]
missing = [str(path.relative_to(RUN_DIR)) for path in required if not path.is_file()]
assert not missing, f'Missing required extension artifacts: {missing}'
files = sorted(path for path in RUN_DIR.rglob('*') if path.is_file())
pd.DataFrame({'relative_path': [str(path.relative_to(RUN_DIR)) for path in files], 'size_bytes': [path.stat().st_size for path in files]}).to_csv(RUN_DIR / 'artifact_manifest.csv', index=False)
archive = Path(shutil.make_archive(str(Path('/kaggle/working') / RUN_NAME), 'zip', root_dir=RUN_DIR))
print('Validated artifacts:', len(required))
print('Archive:', archive)
archive